# Shor's algorithm for any N: Beauregard's 2n+3 qubit circuit with Draper QFT adders

Deliverable 1. Everything on the qubits is built here from `h`, `x`, `p`, `cp`, `cx`, `cswap` (plus `swap`, `measure`, `reset` and `if_test` for the counting side). No library QFT, no library arithmetic, no `Gate.control()`, no `Shor` class.

| Layer | Function | Action |
|---|---|---|
| Fourier register | `qft_noswap`, `iqft_noswap` | qubit i of φ(b) carries phase 2πb/2^(i+1) |
| add | `phi_add` | φ(b) → φ(b + a) |
| controlled-add | `c_phi_add`, `cc_phi_add` | same, if the control(s) are 1 |
| controlled-modular-add | `cc_phi_add_mod` | φ(b) → φ((b + a) mod N) if both controls are 1 |
| modular multiply | `c_mult_mod` | (x, b) → (x, (b + a·x) mod N) |
| controlled-U_a | `controlled_ua` | (x, 0) → (a·x mod N, 0) |
| whole algorithm | `build_shor_circuit` | 2n+3 qubits (`semiclassical`) or t+2n+2 (`full`) |

The phase notebooks load this file with `%run shor_core_generalized.ipynb`; keep all notebooks in one folder. Section numbers in comments (§) refer to the PDF blueprint.

In [1]:
import json
import math
import os
from math import gcd, pi

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister

N = 15                  # composite to factor: odd, not prime, not a prime power
A = 7                   # base: 1 < A < N, gcd(A, N) = 1
T = None                # counting precision t; None -> 2n, which guarantees continued fractions work (§5)
MODE = "semiclassical"  # "semiclassical": Beauregard's 2n+3 qubits, one recycled control qubit
                        # "full": explicit t-qubit counting register + IQFT, t + 2n + 2 qubits
AQFT_MAX_K = None       # None = exact QFTs in the adders; integer k drops rotations pi/2^j with j > k
SKIP_IDENTITY = True    # skip controlled-U when a^(2^j) mod N == 1 (then U is exactly the identity)
RESULTS_DIR = "results"

RUN_CORE_DEMO = globals().get("RUN_CORE_DEMO", True)   # phase notebooks set this to False

## Classical number theory (§1, §5)

Pre-checks remove inputs the quantum step cannot or need not handle (even N, primes, prime powers, lucky gcd). The multiplier constants a^(2^j) mod N come from repeated squaring. `multiplicative_order` is brute force and is used **only** for the ideal benchmark baseline, never to build a circuit.

In [2]:
def is_probable_prime(n):
    """Deterministic Miller-Rabin for every n < 3.3e24 (fixed witness set)."""
    if n < 2:
        return False
    witnesses = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41]
    for p in witnesses:
        if n % p == 0:
            return n == p
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    for w in witnesses:
        x = pow(w, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False
    return True


def integer_root(n, k):
    """Largest integer r with r^k <= n (binary search, exact integers)."""
    lo, hi = 1, 1 << (n.bit_length() // k + 1)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if mid ** k <= n:
            lo = mid
        else:
            hi = mid - 1
    return lo


def classical_prechecks(N, a):
    """Returns (factors, reason) when no quantum step is needed, (None, 'ok') otherwise."""
    if N < 4 or is_probable_prime(N):
        raise ValueError(f"N = {N} is not composite")
    if N % 2 == 0:
        return (2, N // 2), "N is even"
    for k in range(2, N.bit_length() + 1):
        r = integer_root(N, k)
        if r > 1 and r ** k == N:
            return (r, N // r), f"N = {r}^{k} is a perfect power (Shor's splitting step fails on prime powers)"
    if not 1 < a < N:
        raise ValueError("need 1 < a < N")
    g = gcd(a, N)
    if g > 1:
        return (g, N // g), f"gcd(a, N) = {g}: a already shares a factor with N"
    return None, "ok"


def powers_by_squaring(a, N, t):
    """[a^(2^j) mod N for j = 0 .. t-1]: each entry is the square of the previous one."""
    out, m = [], a % N
    for _ in range(t):
        out.append(m)
        m = m * m % N
    return out


def multiplicative_order(a, N):
    """Smallest r > 0 with a^r = 1 mod N, by brute force. Benchmark baseline only."""
    r, v = 1, a % N
    while v != 1:
        v = v * a % N
        r += 1
    return r

## Continued fractions and factor extraction (§5)

`continued_fraction` is Euclid's algorithm on (y, 2^t); `convergents` applies the recurrences h_k = a_k h_(k-1) + h_(k-2), k_k = a_k k_(k-1) + k_(k-2). `find_period` returns the first convergent denominator d < N with a^d ≡ 1 (mod N). `max_multiple > 1` also tries m·d, which rescues outcomes where gcd(s, r) > 1.

In [3]:
def continued_fraction(num, den):
    """Coefficients [a0; a1, a2, ...] of num/den by Euclid's algorithm."""
    coeffs = []
    while den:
        q, rem = divmod(num, den)
        coeffs.append(q)
        num, den = den, rem
    return coeffs


def convergents(coeffs):
    """[(h_k, k_k)] with h_k = a_k h_(k-1) + h_(k-2), k_k = a_k k_(k-1) + k_(k-2), h_-1 = 1, h_-2 = 0, k_-1 = 0, k_-2 = 1."""
    h2, h1, k2, k1 = 0, 1, 1, 0
    out = []
    for c in coeffs:
        h2, h1 = h1, c * h1 + h2
        k2, k1 = k1, c * k1 + k2
        out.append((h1, k1))
    return out


def find_period(y, t, a, N, max_multiple=1):
    """First convergent denominator d < N of y/2^t (or a multiple m*d < N, m <= max_multiple) with a^d = 1 mod N."""
    if y == 0:
        return None
    for _, d in convergents(continued_fraction(y, 2 ** t)):
        if d >= N:
            break
        for m in range(1, max_multiple + 1):
            if m * d < N and pow(a, m * d, N) == 1:
                return m * d
    return None


def factors_from_measurement(y, t, a, N, max_multiple=1):
    """Full classical post-processing of one measured integer y. Returns (factors or None, reason)."""
    if y == 0:
        return None, "y = 0 carries no period information"
    r = find_period(y, t, a, N, max_multiple)
    if r is None:
        return None, "no convergent denominator d < N satisfies a^d = 1 mod N"
    if r % 2:
        return None, f"r = {r} is odd"
    half = pow(a, r // 2, N)
    if half == N - 1:
        return None, f"r = {r} but a^(r/2) = -1 mod N"
    p, q = gcd(half - 1, N), gcd(half + 1, N)
    if {p, q} <= {1, N}:
        return None, f"r = {r} gives only trivial gcds"
    return tuple(sorted((p, q))), (f"r = {r}, a^(r/2) = {half}, gcd({half - 1}, N) = {p}, "
                                   f"gcd({half + 1}, N) = {q}")


def success_rate(counts, t, a, N, max_multiple=1):
    """Fraction of shots (or probability mass) whose outcome yields a nontrivial factor."""
    total = sum(counts.values())
    good = sum(v for bits, v in counts.items()
               if factors_from_measurement(int(bits, 2), t, a, N, max_multiple)[0] is not None)
    return good / total

## Fourier register (§3.1)

Draper's QFT without the final swaps. With `reg[0]` the least significant bit:

QFT_noswap |b⟩ = ⊗_i ( |0⟩ + e^(2πi·b/2^(i+1)) |1⟩ ) / √2 ,

so **qubit i carries the phase 2πb / 2^(i+1)**. The swaps are unnecessary because nobody reads this register in Fourier form; the adders below are written for this order.

In [4]:
def qft_noswap(qc, reg):
    m = len(reg)
    for i in reversed(range(m)):
        qc.h(reg[i])
        for l in reversed(range(i)):
            if AQFT_MAX_K is None or i - l <= AQFT_MAX_K:
                qc.cp(pi / 2 ** (i - l), reg[l], reg[i])


def iqft_noswap(qc, reg):
    m = len(reg)
    for i in range(m):
        for l in range(i):
            if AQFT_MAX_K is None or i - l <= AQFT_MAX_K:
                qc.cp(-pi / 2 ** (i - l), reg[l], reg[i])
        qc.h(reg[i])

## add, controlled-add, doubly-controlled-add (§3.2, §3.3)

Adding a classical constant a in the Fourier basis is one phase gate per qubit: P(2πa/2^(i+1)) on qubit i turns e^(2πi·b/2^(i+1)) into e^(2πi·(b+a)/2^(i+1)). No carries, no ancillas.

The doubly-controlled version uses CCP(θ) = CP(θ/2)[c1] · CX(c1,c2) · CP(−θ/2)[c2] · CX(c1,c2) · CP(θ/2)[c2] on the whole register at once, so it costs 3(n+1) controlled phases and only 2 CNOTs.

In [5]:
def phi_add_angles(a, m):
    """Angles 2*pi*a/2^(i+1), reduced exactly with integers to [0, 2*pi), for qubits i = 0..m-1."""
    return [2 * pi * (a % 2 ** (i + 1)) / 2 ** (i + 1) for i in range(m)]


def _nonzero(theta):
    return abs(theta) > 1e-12


def phi_add(qc, b, a):
    """add:  phi(b) -> phi(b + a mod 2^m)."""
    for i, th in enumerate(phi_add_angles(a, len(b))):
        if _nonzero(th):
            qc.p(th, b[i])


def c_phi_add(qc, ctrl, b, a):
    """controlled-add:  |c>phi(b) -> |c>phi(b + c*a mod 2^m)."""
    for i, th in enumerate(phi_add_angles(a, len(b))):
        if _nonzero(th):
            qc.cp(th, ctrl, b[i])


def cc_phi_add(qc, c1, c2, b, a):
    """doubly-controlled-add:  |c1 c2>phi(b) -> |c1 c2>phi(b + c1*c2*a mod 2^m)."""
    angles = phi_add_angles(a, len(b))
    if not any(_nonzero(th) for th in angles):
        return
    for i, th in enumerate(angles):
        if _nonzero(th):
            qc.cp(th / 2, c1, b[i])
    qc.cx(c1, c2)
    for i, th in enumerate(angles):
        if _nonzero(th):
            qc.cp(-th / 2, c2, b[i])
    qc.cx(c1, c2)
    for i, th in enumerate(angles):
        if _nonzero(th):
            qc.cp(th / 2, c2, b[i])

## controlled-modular-add (§3.4, Beauregard Fig. 5)

Input φ(b) with 0 ≤ a, b < N on n+1 qubits (the extra top qubit is the sign/overflow bit), ancilla |0⟩.

1. add a (doubly controlled), subtract N: φ(b + a − N)
2. IQFT, copy the sign bit into `anc` (anc = 1 iff b + a < N), QFT
3. add N controlled on `anc`: now φ((b + a) mod N)
4. subtract a, IQFT, the sign bit is now NOT anc: flip it, CNOT into anc (anc returns to 0), flip back, QFT
5. add a again: φ((b + a) mod N) with a clean ancilla

In [6]:
def cc_phi_add_mod(qc, c1, c2, b, anc, a, N):
    """controlled-modular-add:  phi(b) -> phi((b + a) mod N) if c1 = c2 = 1, else phi(b); anc |0> -> |0>."""
    msb = b[-1]
    cc_phi_add(qc, c1, c2, b, a)          # phi(b + a)
    phi_add(qc, b, -N)                    # phi(b + a - N)
    iqft_noswap(qc, b)
    qc.cx(msb, anc)                       # anc = 1 iff b + a - N < 0
    qft_noswap(qc, b)
    c_phi_add(qc, anc, b, N)              # phi((b + a) mod N)
    cc_phi_add(qc, c1, c2, b, -a)         # phi((b + a) mod N - a): sign bit = NOT anc
    iqft_noswap(qc, b)
    qc.x(msb)
    qc.cx(msb, anc)                       # anc -> 0
    qc.x(msb)
    qft_noswap(qc, b)
    cc_phi_add(qc, c1, c2, b, a)          # phi((b + a) mod N)

## Modular multiplication and controlled-U_a (§3.5, §3.6, Beauregard Figs. 6 and 7)

`c_mult_mod` writes a·x = Σ_i x_i (2^i a mod N) and performs one controlled-modular-add per bit of x, each controlled by `ctrl` and x_i.

`controlled_ua` makes the multiplication in place: CMULT(a) puts a·x into b, controlled SWAPs exchange x and b, and the inverse of CMULT(a⁻¹) clears b because b − a⁻¹·(a·x) = 0 (mod N). This needs gcd(a, N) = 1.

In [7]:
def c_mult_mod(qc, ctrl, x, b, anc, a, N):
    """|c>|x>|b> -> |c>|x>|(b + c*a*x) mod N>, with 0 <= b < N."""
    qft_noswap(qc, b)
    for i in range(len(x)):
        cc_phi_add_mod(qc, ctrl, x[i], b, anc, (a * 2 ** i) % N, N)
    iqft_noswap(qc, b)


def controlled_ua(qc, ctrl, x, b, anc, a, N):
    """|c>|x>|0>|0> -> |c>|(a^c * x) mod N>|0>|0>, for 0 <= x < N and gcd(a, N) = 1."""
    c_mult_mod(qc, ctrl, x, b, anc, a, N)                 # b = a*x mod N
    for i in range(len(x)):
        qc.cswap(ctrl, x[i], b[i])                        # x <-> b (low n qubits)
    undo = QuantumCircuit(*qc.qregs)                       # same qubits, built separately
    c_mult_mod(undo, ctrl, x, b, anc, pow(a, -1, N), N)
    qc.compose(undo.inverse(), inplace=True)              # b = b - a^-1 * (a*x) = 0

## Counting side and the complete circuit (§2, §4)

* `full`: t counting qubits, Hadamards, count[j] controls U^(2^j), IQFT, measure. Total t + 2n + 2 qubits.
* `semiclassical` (Beauregard's 2n+3): one control qubit reused t times. Step k applies U^(2^(t−1−k)), removes the bits already measured with classically controlled phases −π/2^(k−m), applies H and measures bit k of y. This is the IQFT carried out one qubit at a time (Griffiths and Niu); the output distribution is identical to `full` (§4.4).

Both write bit k of y into classical bit k, so the bitstring read as binary is y.

In [8]:
def iqft_counting(qc, qubits):
    """IQFT with swaps on the counting register: |x> -> M^(-1/2) sum_y exp(-2 pi i x y / M) |y>."""
    n = len(qubits)
    for i in range(n // 2):
        qc.swap(qubits[i], qubits[n - 1 - i])
    for j in range(n):
        for m in range(j):
            qc.cp(-pi / 2 ** (j - m), qubits[m], qubits[j])
        qc.h(qubits[j])


def build_shor_circuit(N, a, t=None, mode="semiclassical", skip_identity=True):
    factors, why = classical_prechecks(N, a)
    if factors is not None:
        raise ValueError(f"No quantum step needed: {why} -> {factors}")
    n = N.bit_length()
    t = 2 * n if t is None else t
    mults = powers_by_squaring(a, N, t)                    # a^(2^j) mod N, computed classically

    x = QuantumRegister(n, "x")                            # work register, starts at |1>
    b = QuantumRegister(n + 1, "b")                        # Fourier accumulator, |0>
    anc = QuantumRegister(1, "anc")                        # modular-adder ancilla, |0>
    c = ClassicalRegister(t, "c")

    if mode == "semiclassical":
        ctrl = QuantumRegister(1, "ctrl")
        qc = QuantumCircuit(ctrl, x, b, anc, c, name=f"shor_N{N}_a{a}_t{t}_semiclassical")
        qc.x(x[0])
        for k in range(t):                                 # step k measures bit k of y (LSB first)
            j = t - 1 - k
            if k > 0:
                qc.reset(ctrl[0])
            qc.h(ctrl[0])
            if not (skip_identity and mults[j] == 1):
                controlled_ua(qc, ctrl[0], x, b, anc[0], mults[j], N)
            for m in range(k):                             # semiclassical IQFT corrections
                with qc.if_test((c[m], 1)):
                    qc.p(-pi / 2 ** (k - m), ctrl[0])
            qc.h(ctrl[0])
            qc.measure(ctrl[0], c[k])
    elif mode == "full":
        count = QuantumRegister(t, "count")
        qc = QuantumCircuit(count, x, b, anc, c, name=f"shor_N{N}_a{a}_t{t}_full")
        qc.x(x[0])
        for q in count:
            qc.h(q)
        for j in range(t):
            if not (skip_identity and mults[j] == 1):
                controlled_ua(qc, count[j], x, b, anc[0], mults[j], N)
        iqft_counting(qc, list(count))
        for j in range(t):
            qc.measure(count[j], c[j])
    else:
        raise ValueError("mode must be 'semiclassical' or 'full'")
    return qc

## Analysis helpers (used by Phases A, B and C)

`exact_distribution` is the closed form of §4.3: the ideal output distribution of the circuit, computed classically in O(2^t). It is the zero-noise baseline for the benchmarks. The demo below checks it against a noiseless simulation of the actual circuit.

In [9]:
def exact_distribution(a, N, t):
    """P(y) = (1/M^2) [ n_hi F(K_hi, y) + (r - n_hi) F(K_lo, y) ],  F(K, y) = sin^2(K theta/2) / sin^2(theta/2),
    theta = 2 pi r y / M,  K_lo = floor(M/r),  K_hi = K_lo + 1,  n_hi = M mod r."""
    import numpy as np
    M = 2 ** t
    r = multiplicative_order(a, N)
    y = np.arange(M)
    theta = 2 * np.pi * ((r * y) % M) / M
    s = np.sin(theta / 2)
    peak = np.abs(s) < 1e-12

    def F(K):
        return np.where(peak, float(K * K), np.sin(K * theta / 2) ** 2 / np.where(peak, 1.0, s ** 2))

    K_lo, n_hi = M // r, M % r
    P = (n_hi * F(K_lo + 1) + (r - n_hi) * F(K_lo)) / M ** 2
    return {format(int(v), f"0{t}b"): float(p) for v, p in zip(y, P) if p > 1e-15}


def peak_outcomes(a, N, t):
    """Integers nearest to s*M/r, s = 0..r-1 (where the ideal probability concentrates)."""
    M, r = 2 ** t, multiplicative_order(a, N)
    return sorted({round(s * M / r) % M for s in range(r)})


def hellinger_fidelity(p, q):
    """F = (sum_k sqrt(p_k q_k))^2 after normalising both. Same definition as qiskit.quantum_info.hellinger_fidelity."""
    sp, sq = sum(p.values()), sum(q.values())
    bc = sum(math.sqrt((p[k] / sp) * (q[k] / sq)) for k in p.keys() & q.keys())
    return bc ** 2


def two_qubit_gate_count(qc):
    return sum(1 for inst in qc.data
               if len(inst.qubits) == 2 and inst.operation.name not in ("barrier", "measure"))


def circuit_report(qc):
    return {"num_qubits_declared": qc.num_qubits, "depth": qc.depth(),
            "two_qubit_gates": two_qubit_gate_count(qc), "ops": dict(qc.count_ops())}


def best_isa_circuit(qc, backend, seeds=range(3)):
    """optimization_level=3 with several layout/routing seeds; keep the fewest two-qubit gates."""
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    best = None
    for s in seeds:
        pm = generate_preset_pass_manager(backend=backend, optimization_level=3, seed_transpiler=s)
        isa = pm.run(qc)
        score = (two_qubit_gate_count(isa), isa.depth())
        if best is None or score < best[0]:
            best = (score, isa, s)
    return best[1], best[2]


def ensure_control_flow(backend):
    """Some fake (snapshot) backends omit dynamic-circuit instructions from their Target.
    AerSimulator executes them natively, so declaring them lets the transpiler accept the circuit."""
    from qiskit.circuit import IfElseOp, Reset
    names = backend.target.operation_names
    if "if_else" not in names:
        backend.target.add_instruction(IfElseOp, name="if_else")
    if "reset" not in names:
        backend.target.add_instruction(Reset(), {(q,): None for q in range(backend.num_qubits)})


def run_tag(N, a, t, mode):
    return f"N{N}_a{a}_t{t}_{mode}"


def results_path(name):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    return os.path.join(RESULTS_DIR, name)


def save_json(name, obj):
    with open(results_path(name), "w") as f:
        json.dump(obj, f, indent=2)


def load_json(name):
    path = results_path(name)
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

## Demo 1: pre-checks and resources (§1)

Cells from here on are skipped when a phase notebook loads this one.

In [10]:
if RUN_CORE_DEMO:
    print("pre-checks:", classical_prechecks(N, A))
    n = N.bit_length()
    t = 2 * n if T is None else T
    qc = build_shor_circuit(N, A, T, MODE, SKIP_IDENTITY)
    print(f"n = {n}, t = {t}, M = 2^t = {2 ** t} >= N^2 = {N * N}: {2 ** t >= N * N}")
    print(f"a^(2^j) mod N, j = 0..{t - 1}: {powers_by_squaring(A, N, t)}")
    print(f"qubits: {qc.num_qubits}  ({'2n+3' if MODE == 'semiclassical' else 't+2n+2'} = "
          f"{2 * n + 3 if MODE == 'semiclassical' else t + 2 * n + 2})")
    print(f"logical ops: {dict(qc.count_ops())}")
    print(f"two-qubit gates (cswap counted separately): {two_qubit_gate_count(qc)}")

pre-checks: (None, 'ok')
n = 4, t = 8, M = 2^t = 256 >= N^2 = 225: True
a^(2^j) mod N, j = 0..7: [7, 4, 1, 1, 1, 1, 1, 1]
qubits: 11  (2n+3 = 11)
logical ops: {'cp': 1394, 'h': 376, 'cx': 128, 'p': 80, 'x': 33, 'if_else': 28, 'measure': 8, 'cswap': 8, 'reset': 7}
two-qubit gates (cswap counted separately): 1522


## Demo 2: arithmetic self-test on basis states (§3)

Exact state-vector checks of `phi_add` for every input, `cc_phi_add_mod` for every b and a sample of a, and `controlled_ua` for every x < N with the control on and off.

In [11]:
if RUN_CORE_DEMO:
    import random
    import numpy as np
    from qiskit.quantum_info import Statevector

    def basis_out(circ):
        """Index of the output basis state; asserts the output is a single basis state."""
        amps = np.abs(Statevector(circ).data)
        k = int(np.argmax(amps))
        assert abs(amps[k] - 1) < 1e-6, "output is not a basis state"
        return k

    def load(circ, qubits, value):
        for i, q in enumerate(qubits):
            if (value >> i) & 1:
                circ.x(q)

    n = N.bit_length()
    m = n + 1

    # add: phi(b) -> phi(b + a mod 2^m)
    for a_test in (1, 5, -3, N):
        for b_val in range(2 ** m):
            reg = QuantumRegister(m, "b")
            circ = QuantumCircuit(reg)
            load(circ, reg, b_val)
            qft_noswap(circ, reg)
            phi_add(circ, reg, a_test)
            iqft_noswap(circ, reg)
            assert basis_out(circ) == (b_val + a_test) % 2 ** m
    print("add: OK")

    # controlled-modular-add for every b < N, a sample of a, controls 11 / 10 / 01
    for a_test in random.Random(1).sample(range(N), min(4, N)):
        for b_val in range(N):
            for c1v, c2v in ((1, 1), (1, 0), (0, 1)):
                c1, c2 = QuantumRegister(1, "c1"), QuantumRegister(1, "c2")
                reg, an = QuantumRegister(m, "b"), QuantumRegister(1, "anc")
                circ = QuantumCircuit(c1, c2, reg, an)
                load(circ, c1, c1v); load(circ, c2, c2v); load(circ, reg, b_val)
                qft_noswap(circ, reg)
                cc_phi_add_mod(circ, c1[0], c2[0], reg, an[0], a_test, N)
                iqft_noswap(circ, reg)
                out = basis_out(circ)
                expect = (b_val + a_test) % N if c1v and c2v else b_val
                assert (out >> 2) & (2 ** m - 1) == expect and out >> (2 + m) == 0
    print("controlled-modular-add: OK")

    # controlled-U_a for every x < N
    for x_val in range(N):
        for cv in (0, 1):
            ct, xr = QuantumRegister(1, "ctrl"), QuantumRegister(n, "x")
            br, an = QuantumRegister(m, "b"), QuantumRegister(1, "anc")
            circ = QuantumCircuit(ct, xr, br, an)
            load(circ, ct, cv); load(circ, xr, x_val)
            controlled_ua(circ, ct[0], xr, br, an[0], A, N)
            out = basis_out(circ)
            expect = (A * x_val) % N if cv else x_val
            assert (out >> 1) & (2 ** n - 1) == expect and out >> (1 + n) == 0
    print(f"controlled-U_a (a = {A}, N = {N}) for all x < N: OK")

add: OK
controlled-modular-add: OK
controlled-U_a (a = 7, N = 15) for all x < N: OK


## Demo 3: noiseless run of the full algorithm vs the exact distribution (§4, §5)

Hellinger fidelity close to 1 (limited only by finite shots) confirms that the circuit produces the distribution derived in §4.3.

In [12]:
if RUN_CORE_DEMO:
    from qiskit import transpile
    from qiskit_aer import AerSimulator

    DEMO_SHOTS = 4096
    sim = AerSimulator()
    counts = sim.run(transpile(qc, sim), shots=DEMO_SHOTS, seed_simulator=7).result().get_counts()
    ideal = exact_distribution(A, N, t)
    print(f"Hellinger fidelity, noiseless circuit vs exact formula: {hellinger_fidelity(ideal, counts):.4f}")
    print(f"success rate: sampled {success_rate(counts, t, A, N):.3f}, exact {success_rate(ideal, t, A, N):.3f}")

Hellinger fidelity, noiseless circuit vs exact formula: 1.0000
success rate: sampled 0.506, exact 0.500


In [13]:
if RUN_CORE_DEMO:
    M = 2 ** t
    print(f"{'y':>6} {'shots':>6} {'y/M':>8}  continued fraction -> convergents -> outcome")
    for bits, cnt in sorted(counts.items(), key=lambda kv: -kv[1])[:10]:
        y = int(bits, 2)
        cf = continued_fraction(y, M)
        conv = [f"{h}/{k}" for h, k in convergents(cf) if k < N]
        facs, why = factors_from_measurement(y, t, A, N)
        print(f"{y:>6} {cnt:>6} {y / M:8.5f}  {cf} -> {conv} -> {'FACTORS ' + str(facs) if facs else 'fail'}: {why}")

     y  shots      y/M  continued fraction -> convergents -> outcome
    64   1040  0.25000  [0, 4] -> ['0/1', '1/4'] -> FACTORS (3, 5): r = 4, a^(r/2) = 4, gcd(3, N) = 3, gcd(5, N) = 5
   192   1031  0.75000  [0, 1, 3] -> ['0/1', '1/1', '3/4'] -> FACTORS (3, 5): r = 4, a^(r/2) = 4, gcd(3, N) = 3, gcd(5, N) = 5
   128   1023  0.50000  [0, 2] -> ['0/1', '1/2'] -> fail: no convergent denominator d < N satisfies a^d = 1 mod N
     0   1002  0.00000  [0] -> ['0/1'] -> fail: y = 0 carries no period information
